# 01 — MaxViT-512 classifier

This notebook implements the complete MaxViT-512 arm of the downstream classification study. A
single execution traverses the four protocol-declared training conditions and seeds 17, 42, and 73,
yielding 12 independent, resumable jobs under a common optimization policy. Model construction,
optimization, validation, checkpoint selection, calibration, error analysis, and attribution are
kept explicit so that the computational path from the input datasets to the reported validation
artifacts remains auditable. The held-out test split is not opened anywhere in this notebook.

## 1. Resolve the experimental matrix and immutable policy

The configuration cell locates the repository root, imports the canonical classifier protocol, and
materializes the 4 × 3 condition–seed matrix for the `maxvit512` architecture. Each job receives a
policy signature, a dataset variant, and an isolated results directory. Synthetic conditions are
resolved through the committed generator selection, while real-only conditions remain
independent of generator artifacts; these identifiers later form part of the checkpoint
compatibility boundary.

In [ ]:
from pathlib import Path
import sys
import os

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))

from notebooks.utility.classifier_experiment import (
    build_error_case_table,
    configure_environment,
    construct_dataset,
    experiment_configuration,
    load_adapter,
    load_existing_outputs,
    load_prediction_rows,
    plot_calibration,
    plot_source_accounting,
    plot_training_history,
    plot_validation_curves,
    run_validation,
    title_classifier_figure,
)

ARCHITECTURE = 'maxvit512'
CONDITIONS = (
    'real_only',
    'real_augmented',
    'real_plus_best_finetuned_positive',
    'real_plus_best_fromscratch_positive',
)
SEEDS = (17, 42, 73)
TRAINING_VERBOSE = True
PROGRESS_EVERY_UPDATES = 25  # use 1 to report every optimizer update
VALIDATION_PROGRESS_EVERY_BATCHES = 10

_CLASSIFIER_GPU = os.environ.get('CLASSIFIER_GPU', 'auto')
configurations = {}
for condition in CONDITIONS:
    configurations[condition] = {}
    for seed in SEEDS:
        job_configuration = experiment_configuration(
            ROOT, ARCHITECTURE, condition, seed, gpu=_CLASSIFIER_GPU
        )
        job_configuration['root'] = str(ROOT)
        configurations[condition][seed] = job_configuration

# Compatibility alias used only by the shared GPU-configuration cell below.
configuration = configurations[CONDITIONS[0]][SEEDS[0]]
{
    condition: {
        seed: {
            'experiment_id': configurations[condition][seed]['experiment_id'],
            'results_dir': configurations[condition][seed]['results_dir'],
        }
        for seed in SEEDS
    }
    for condition in CONDITIONS
}


## 2. Configure a portable and verified compute environment

Before model code is imported, the runtime selects a GPU from the host inventory using
`CLASSIFIER_GPU` (default: `auto`) and verifies the selected physical identity after CUDA
initialization. Automatic mode chooses the device with the greatest reported memory and records the
resolved UUID as runtime metadata rather than embedding a machine-specific identifier in the
notebook. The same verified environment is attached to every job configuration, and execution
refuses ambiguous or inconsistent device mappings instead of silently changing hardware.

In [ ]:
environment = configure_environment(configuration)
for condition in CONDITIONS:
    for seed in SEEDS:
        job_configuration = configurations[condition][seed]
        job_configuration['gpu_uuid'] = environment['resolved_uuid']
        job_configuration['gpu_name'] = environment['observed_name']
        job_configuration['gpu_physical_index'] = environment['resolved_physical_index']
environment_for_display = {
    key: ('<runtime-resolved>' if key in {'resolved_uuid', 'CUDA_VISIBLE_DEVICES'} else value)
    for key, value in environment.items()
}
environment_for_display


## 3. Construct the four training datasets

For every condition and seed, the dataset builder reads the canonical processed metadata and
assembles the corresponding training rows: real only, real plus traditional augmentation, real plus
selected fine-tuned synthetic positives, or real plus selected from-scratch synthetic positives.
Validation rows always come from the unchanged real validation split. Synthetic samples are consumed
from the selected generator's canonical FILTERED positive pool, verified for the exact 1,361 unique
images, and no augmentation or synthetic data are introduced into validation.

In [ ]:
datasets = {
    condition: {
        seed: construct_dataset(ROOT, configurations[condition][seed])
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
{
    condition: {
        seed: {
            'train_rows': len(datasets[condition][seed]['train_rows']),
            'validation_rows': len(datasets[condition][seed]['validation_rows']),
        }
        for seed in SEEDS
    }
    for condition in CONDITIONS
}


## 4. Audit composition, accounting, and patient separation

This read-only audit exposes class and source counts, source accounting, and the overlap check between
training and validation patients for all 12 jobs. It makes the effective intervention in each
condition visible before optimization begins. Any forbidden test path, duplicate file,
missing sample, or patient leakage is treated as a hard error rather than a
warning.

In [ ]:
{
    condition: {
        seed: datasets[condition][seed]['audit']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}


## 5. Define the shared training and evaluation primitives

The following code imports numerical, checkpoint, and metric utilities and defines the explicit
per-job training routine. It fixes the source-accounting schema, deterministic seeding, binary
ROC-AUC/PR-AUC calculation, atomic history persistence, fixed-update data loading, and checkpoint
compatibility fields. These primitives ensure that jobs with different dataset sizes receive the
same optimizer-update budget and that every consumed source type is counted across resumes.

In [ ]:
import csv
import gc
import hashlib
import math
import os
import random
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, roc_auc_score

from notebooks.utility import classifier_checkpoint_io as checkpoint_io
from notebooks.utility.classifier_protocol import atomic_json


## 6. Train or resume all condition–seed jobs

Each job starts from a freshly constructed MaxViT model, freezes the backbone selectively, unfreezes
the classification head and final stages, and optimizes trainable parameters with AdamW, binary focal
loss, mixed precision when available, and a validation-PR-AUC scheduler. Validation occurs at fixed
optimizer-update boundaries; the best checkpoint is selected by maximum validation PR-AUC, with
validation loss used only as the declared tie-break. Resume files contain model, optimizer,
scheduler, scaler, random-number-generator, progress, and source-accounting state, and are accepted
only at a completed validation boundary. Runs carrying a compatible completion marker are verified
and reused without allocating a model.

In [ ]:
def train_one_job(condition, seed, configuration, dataset):
    """Build a fresh state and complete or resume one condition/seed job."""
    CONDITION = str(condition)
    SEED = int(seed)
    POLICY = configuration['policy']
    RUN_DIR = Path(configuration['results_dir'])
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR = Path(configuration['checkpoint_dir'])
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    if PROGRESS_EVERY_UPDATES < 1 or VALIDATION_PROGRESS_EVERY_BATCHES < 1:
        raise ValueError('Verbose progress intervals must be positive integers.')
    SOURCE_ACCOUNTING_FIELDS = (
        'real_negative_seen',
        'real_positive_seen',
        'traditional_augmented_seen',
        'finetuned_synthetic_seen',
        'fromscratch_synthetic_seen',
    )
    HISTORY_FIELDS = (
        'loss', 'auc', 'pr_auc', 'val_loss', 'val_auc', 'val_pr_auc',
        'learning_rate', 'optimizer_steps',
    )


    def seed_everything(seed, device):
        random.seed(seed)
        np.random.seed(seed)
        torch.default_generator.manual_seed(seed)
        if device.type == 'cuda':
            torch.cuda.manual_seed(seed)


    def accounting_field(row):
        source = str(row.get('source', '')).lower()
        if source == 'augmented':
            return 'traditional_augmented_seen'
        if source == 'synthetic':
            family = str(row.get('synthetic_family', '')).lower()
            if family == 'finetuned':
                return 'finetuned_synthetic_seen'
            if family == 'from_scratch':
                return 'fromscratch_synthetic_seen'
            raise ValueError(f"Invalid synthetic family: {family!r}")
        return 'real_positive_seen' if int(row['label']) == 1 else 'real_negative_seen'


    def accounting_metadata(rows):
        return [
            {
                'sample_id': str(row.get('image_id') or row.get('sample_id') or index),
                'source': str(row.get('source', 'unknown')),
                'accounting_field': accounting_field(row),
            }
            for index, row in enumerate(rows)
        ]


    class FixedBatchLoader:
        """Repeat the shuffled loader until one fixed validation block is complete."""

        def __init__(self, loader, batch_count):
            self.loader = loader
            self.batch_count = int(batch_count)

        def __len__(self):
            return self.batch_count

        def __iter__(self):
            emitted = 0
            while emitted < self.batch_count:
                cycle_count = 0
                for batch in self.loader:
                    yield batch
                    emitted += 1
                    cycle_count += 1
                    if emitted >= self.batch_count:
                        return
                if cycle_count == 0:
                    raise RuntimeError('Training loader is empty.')


    def binary_metrics(labels, probabilities):
        try:
            roc_auc = float(roc_auc_score(labels, probabilities))
        except ValueError:
            roc_auc = float('nan')
        try:
            pr_auc = float(average_precision_score(labels, probabilities))
        except ValueError:
            pr_auc = float('nan')
        return {'auc': roc_auc, 'pr_auc': pr_auc}


    def accounting_snapshot():
        return {
            'schema_version': 1,
            'accounting_mode': 'actual',
            **source_counts,
            'total_samples_seen': sum(source_counts.values()),
        }


    def write_history_csv(path, values):
        rows = [
            {
                'epoch': index + 1,
                **{
                    field: values[field][index] if index < len(values[field]) else None
                    for field in HISTORY_FIELDS
                },
            }
            for index in range(max((len(values[field]) for field in HISTORY_FIELDS), default=0))
        ]
        temporary = path.with_name(path.name + f'.tmp.{os.getpid()}')
        with temporary.open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=['epoch', *HISTORY_FIELDS])
            writer.writeheader()
            writer.writerows(rows)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)


    def normalized_gpu_uuid(value):
        return str(value or '').strip().lower().removeprefix('gpu-')


    validation_interval = int(POLICY['validation_interval_updates'])
    max_optimizer_updates = int(POLICY['max_optimizer_updates'])
    epochs = min(
        int(POLICY.get('max_epochs_secondary_limit', 60)),
        max(1, math.ceil(max_optimizer_updates / validation_interval)),
    )
    early_stopping_patience = int(POLICY['early_stopping']['patience'])
    completion_limits = {
        'max_optimizer_updates': max_optimizer_updates,
        'max_epochs': epochs,
        'early_stopping_patience': early_stopping_patience,
    }
    expected_checkpoint = {
        'architecture': ARCHITECTURE,
        'experiment_id': configuration['experiment_id'],
        'dataset_variant_id': CONDITION,
        'training_policy': configuration['training_policy_name'],
        'config_signature': configuration['policy_signature'],
        'dataset_signature': dataset['dataset_metadata']['signature'],
        'seed': int(SEED),
    }
    completed_run, completion_source = checkpoint_io.inspect_completed_run(
        RUN_DIR, CHECKPOINT_DIR, expected_checkpoint, completion_limits
    )
    if completed_run is not None:
        if completion_source != checkpoint_io.COMPLETION_NAME:
            atomic_json(checkpoint_io.completion_path(RUN_DIR), completed_run)
        existing_outputs = load_existing_outputs(ROOT, configuration)
        checkpoint_gpu_uuid = completed_run.get('checkpoint_gpu_uuid')
        runtime_gpu_uuid = configuration.get('gpu_uuid')
        gpu_changed = bool(
            checkpoint_gpu_uuid and runtime_gpu_uuid
            and normalized_gpu_uuid(checkpoint_gpu_uuid) != normalized_gpu_uuid(runtime_gpu_uuid)
        )
        print(
            f'SKIP completed run | reason={completed_run["completion_reason"]} | '
            f'step={completed_run["optimizer_updates_completed"]}/'
            f'{max_optimizer_updates}',
            flush=True,
        )
        return {
            'configuration': configuration,
            'dataset': dataset,
            'training_result': {
                'status': 'already_complete',
                'completion_reason': completed_run['completion_reason'],
                'completion_source': completion_source,
                'checkpoint': existing_outputs['checkpoint'],
                'history': existing_outputs['history'],
                'resumed_from': None,
                'optimizer_updates_limit': max_optimizer_updates,
                'optimizer_updates_completed': completed_run[
                    'optimizer_updates_completed'
                ],
                'epochs': epochs,
                'best_epoch': completed_run.get('best_epoch'),
                'source_accounting': existing_outputs['source_accounting'],
                'output_dir': str(RUN_DIR),
                'checkpoint_gpu_uuid': checkpoint_gpu_uuid,
                'runtime_gpu_uuid': runtime_gpu_uuid,
                'gpu_changed': gpu_changed,
            },
        }

    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cuda':
        try:
            torch.cuda.set_device(device)
            torch.cuda.synchronize(device)
            torch.cuda.empty_cache()
        except Exception as exc:
            raise RuntimeError(
                'CUDA context is not healthy. Restart only this notebook kernel '
                'before resuming the classifier sweep.'
            ) from exc
    seed_everything(SEED, device)
    os.environ.setdefault('HF_HUB_OFFLINE', '1')
    from notebooks.utility import maxvit_utils as architecture_utils

    common_utils = architecture_utils
    model = architecture_utils.build_maxvit_model(num_classes=1, pretrained=True)
    architecture_utils.freeze_all(model)
    architecture_utils.unfreeze_head(model)
    architecture_utils.unfreeze_stages_from(
        model, max(0, len(model.stages) - 2)
    )

    train_frame = pd.DataFrame(dataset['train_rows'])
    validation_frame = pd.DataFrame(dataset['validation_rows'])
    batch_size = int(POLICY['physical_batch_size'])
    workers = int(POLICY.get('dataloader_workers', 0))
    mean = POLICY['normalization']['mean']
    std = POLICY['normalization']['std']
    image_size = int(POLICY['input_size'][0])
    base_train_loader = architecture_utils.make_dataloader(
        train_frame, 'processed_path', 'label', mean, std, image_size,
        batch_size, True, True, SEED, workers,
        metadata=accounting_metadata(dataset['train_rows']),
    )
    validation_loader = architecture_utils.make_dataloader(
        validation_frame, 'processed_path', 'label', mean, std, image_size,
        batch_size, False, False, SEED, workers,
    )

    accumulation_steps = int(POLICY.get('gradient_accumulation_steps', 1))
    train_loader = FixedBatchLoader(
        base_train_loader,
        validation_interval * accumulation_steps,
    )

    model.to(device)
    parameters_to_optimize = [
        parameter for parameter in model.parameters() if parameter.requires_grad
    ]
    optimizer = torch.optim.AdamW(
        parameters_to_optimize,
        lr=float(POLICY['training_phases'][0]['learning_rate']),
        weight_decay=float(POLICY.get('weight_decay', 0.0)),
    )
    criterion = common_utils.BinaryFocalLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=float(POLICY['scheduler_params']['factor']),
        patience=int(POLICY['scheduler_params']['patience']),
        min_lr=float(POLICY['scheduler_params']['min_lr']),
    )
    scaler = (
        torch.amp.GradScaler('cuda')
        if bool(POLICY.get('amp')) and device.type == 'cuda'
        else None
    )

    resume, resume_source = checkpoint_io.load_resume_checkpoint(
        CHECKPOINT_DIR, expected_checkpoint
    )
    checkpoint_files_exist = any(CHECKPOINT_DIR.glob('checkpoint_*'))
    if resume is None and resume_source == 'no resume checkpoint' and checkpoint_files_exist:
        raise RuntimeError(
            f'Checkpoint files exist in {CHECKPOINT_DIR}, but none is resumable. '
            'Move the run directory before starting again from zero.'
        )
    if resume is None and resume_source != 'no resume checkpoint':
        raise RuntimeError(f'Corrupt or incompatible resume checkpoint: {resume_source}')

    required_resume_fields = {
        'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict',
        'epoch', 'global_step', 'rng_states', 'source_accounting',
    }
    if resume is not None:
        missing = sorted(required_resume_fields - set(resume))
        if missing:
            raise RuntimeError(f'Incomplete resume checkpoint: {missing}')
        if int(resume.get('batch_index', -1)) != -1:
            raise RuntimeError('Resume is allowed only at a validated epoch boundary.')
        if resume['source_accounting'].get('accounting_mode') != 'actual':
            raise RuntimeError('Resume checkpoint lacks actual source accounting.')

    runtime_gpu_uuid = configuration.get('gpu_uuid')
    checkpoint_gpu_uuid = resume.get('gpu_uuid') if resume else None
    gpu_execution_metadata = {
        'checkpoint_gpu_uuid': checkpoint_gpu_uuid,
        'runtime_gpu_uuid': runtime_gpu_uuid,
        'gpu_changed': bool(
            checkpoint_gpu_uuid
            and runtime_gpu_uuid
            and normalized_gpu_uuid(checkpoint_gpu_uuid) != normalized_gpu_uuid(runtime_gpu_uuid)
        ),
    }

    start_epoch = 1
    global_step = 0
    best_pr_auc = float('-inf')
    best_validation_loss = float('inf')
    best_epoch = None
    early_stopping_wait = 0
    history = {field: [] for field in HISTORY_FIELDS}
    source_counts = {field: 0 for field in SOURCE_ACCOUNTING_FIELDS}

    if resume is not None:
        model.load_state_dict(resume['model_state_dict'], strict=True)
        optimizer.load_state_dict(resume['optimizer_state_dict'])
        scheduler.load_state_dict(resume['scheduler_state_dict'])
        if scaler is not None and resume.get('scaler_state_dict'):
            scaler.load_state_dict(resume['scaler_state_dict'])
        start_epoch = int(resume['epoch'])
        global_step = int(resume['global_step'])
        best_pr_auc = float(resume.get('best_metric', float('-inf')))
        best_validation_loss = float(
            resume.get('best_validation_loss', float('inf'))
        )
        best_epoch = resume.get('best_epoch')
        early_stopping_wait = int(resume.get('early_stopping_counter', 0))
        prior_history = resume.get('history', {})
        history = {
            field: list(prior_history.get(field, [])) for field in HISTORY_FIELDS
        }
        source_counts.update({
            field: int(resume['source_accounting'].get(field, 0))
            for field in SOURCE_ACCOUNTING_FIELDS
        })
        rng_states = resume['rng_states']
        if rng_states.get('python'):
            random.setstate(rng_states['python'])
        if rng_states.get('numpy'):
            np.random.set_state(rng_states['numpy'])
        if rng_states.get('torch') is not None:
            torch.set_rng_state(rng_states['torch'])
        if torch.cuda.is_available() and rng_states.get('torch_cuda'):
            torch.cuda.set_rng_state_all(rng_states['torch_cuda'])
        for state_key in (
            'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict',
            'scaler_state_dict', 'rng_states',
        ):
            resume.pop(state_key, None)
        del prior_history, rng_states

    atomic_json(
        RUN_DIR / 'model_summary.json',
        {
            'architecture': ARCHITECTURE,
            'parameters': sum(parameter.numel() for parameter in model.parameters()),
            'trainable_parameters': sum(
                parameter.numel()
                for parameter in model.parameters()
                if parameter.requires_grad
            ),
            'input_size': POLICY['input_size'],
        },
    )
    warmup_updates = int(POLICY.get('warmup_updates', 0))
    target_learning_rate = float(POLICY['training_phases'][0]['learning_rate'])
    gradient_clip = POLICY.get('gradient_clipping')
    resume_segment_id = hashlib.sha256(os.urandom(16)).hexdigest()[:16]
    session_start_step = global_step
    session_start_time = time.perf_counter()


    def readable_duration(seconds):
        if seconds is None or not math.isfinite(seconds):
            return 'unknown'
        seconds = max(0, int(seconds))
        hours, remainder = divmod(seconds, 3600)
        minutes, seconds = divmod(remainder, 60)
        return f'{hours:02d}:{minutes:02d}:{seconds:02d}'


    if TRAINING_VERBOSE:
        print(
            f'Starting {ARCHITECTURE} | condition={CONDITION} | seed={SEED} | '
            f'device={device} | resume={resume_source} | '
            f'step={global_step}/{max_optimizer_updates}',
            flush=True,
        )


    def save_training_checkpoint(next_epoch, improved):
        payload = {
            **expected_checkpoint,
            **gpu_execution_metadata,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'epoch': int(next_epoch),
            'batch_index': -1,
            'global_step': int(global_step),
            'checkpoint_metric': 'val_pr_auc',
            'best_metric': float(best_pr_auc),
            'best_validation_loss': float(best_validation_loss),
            'best_epoch': best_epoch,
            'early_stopping_counter': int(early_stopping_wait),
            'history': history,
            'source_accounting': accounting_snapshot(),
            'gpu_uuid': runtime_gpu_uuid,
            'rng_states': {
                'python': random.getstate(),
                'numpy': np.random.get_state(),
                'torch': torch.get_rng_state(),
                'torch_cuda': (
                    torch.cuda.get_rng_state_all()
                    if torch.cuda.is_available()
                    else []
                ),
            },
            'resume_segment_id': resume_segment_id,
        }
        checkpoint_io.save_resume_checkpoint(
            CHECKPOINT_DIR, payload, best=bool(improved)
        )


    completion_reason = checkpoint_io.terminal_reason(
        resume, completion_limits
    )
    if completion_reason is not None:
        print(
            f'Resume checkpoint is already terminal ({completion_reason}); '
            'finalizing artifacts without another training epoch.',
            flush=True,
        )


    for epoch in range(start_epoch, epochs + 1):
        if completion_reason is not None:
            break
        if global_step >= max_optimizer_updates:
            completion_reason = 'max_optimizer_updates'
            break
        if TRAINING_VERBOSE:
            print(
                f'\nEpoch {epoch}/{epochs} started at optimizer step '
                f'{global_step}/{max_optimizer_updates}.',
                flush=True,
            )

        # ---- Training batches: forward, focal loss, backward, accumulation, AdamW.
        model.train()
        common_utils.refreeze_batchnorm(model)
        optimizer.zero_grad(set_to_none=True)
        train_loss_sum = 0.0
        train_seen = 0
        train_labels = []
        train_probabilities = []

        for batch_index, batch in enumerate(train_loader):
            if len(batch) != 3:
                raise RuntimeError(
                    'Training batches must include source-accounting metadata.'
                )
            images, labels, metadata = batch
            images = images.to(device)
            labels = labels.to(device)

            with torch.autocast(
                device_type=device.type,
                enabled=(scaler is not None and device.type == 'cuda'),
            ):
                logits = model(images).squeeze(-1)
                full_loss = criterion(logits, labels)
                loss = full_loss / accumulation_steps

            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            for field in metadata['accounting_field']:
                if field not in source_counts:
                    raise RuntimeError(f'Unknown accounting field: {field}')
                source_counts[field] += 1

            train_loss_sum += float(full_loss.detach()) * images.size(0)
            train_seen += images.size(0)
            train_labels.extend(labels.detach().cpu().numpy())
            train_probabilities.extend(
                torch.sigmoid(logits).detach().cpu().numpy()
            )

            optimizer_boundary = (
                (batch_index + 1) % accumulation_steps == 0
                or (batch_index + 1) == len(train_loader)
            )
            if optimizer_boundary:
                next_step = global_step + 1
                if warmup_updates and next_step <= warmup_updates:
                    warmup_lr = target_learning_rate * next_step / warmup_updates
                    for group in optimizer.param_groups:
                        group['lr'] = warmup_lr

                if gradient_clip is not None:
                    if scaler is not None:
                        scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        parameters_to_optimize, float(gradient_clip)
                    )

                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                global_step = next_step

                report_progress = (
                    global_step == session_start_step + 1
                    or global_step % PROGRESS_EVERY_UPDATES == 0
                    or global_step >= max_optimizer_updates
                )
                if TRAINING_VERBOSE and report_progress:
                    elapsed = time.perf_counter() - session_start_time
                    session_updates = max(global_step - session_start_step, 1)
                    seconds_per_update = elapsed / session_updates
                    eta = seconds_per_update * (max_optimizer_updates - global_step)
                    progress = 100.0 * global_step / max_optimizer_updates
                    print(
                        f'  train | epoch {epoch}/{epochs} | '
                        f'batch {batch_index + 1}/{len(train_loader)} | '
                        f'step {global_step}/{max_optimizer_updates} '
                        f'({progress:5.1f}%) | loss={float(full_loss.detach()):.4f} | '
                        f'lr={optimizer.param_groups[0]["lr"]:.3e} | '
                        f'elapsed={readable_duration(elapsed)} | '
                        f'ETA={readable_duration(eta)}',
                        flush=True,
                    )

                if global_step >= max_optimizer_updates:
                    break

        train_scores = binary_metrics(
            np.asarray(train_labels), np.asarray(train_probabilities)
        )
        train_metrics = {
            'loss': train_loss_sum / max(train_seen, 1),
            **train_scores,
        }

        # ---- Full validation pass: no gradients and no parameter updates.
        model.eval()
        validation_loss_sum = 0.0
        validation_seen = 0
        validation_labels = []
        validation_probabilities = []
        if TRAINING_VERBOSE:
            print(f'  validation | epoch {epoch}/{epochs} started', flush=True)
        with torch.no_grad():
            for validation_batch_index, (images, labels) in enumerate(validation_loader):
                images = images.to(device)
                labels = labels.to(device)
                logits = model(images).squeeze(-1)
                validation_loss = criterion(logits, labels)
                validation_loss_sum += (
                    float(validation_loss.detach()) * images.size(0)
                )
                validation_seen += images.size(0)
                validation_labels.extend(labels.cpu().numpy())
                validation_probabilities.extend(
                    torch.sigmoid(logits).cpu().numpy()
                )
                report_validation = (
                    (validation_batch_index + 1) % VALIDATION_PROGRESS_EVERY_BATCHES == 0
                    or (validation_batch_index + 1) == len(validation_loader)
                )
                if TRAINING_VERBOSE and report_validation:
                    print(
                        f'  validation | batch {validation_batch_index + 1}/'
                        f'{len(validation_loader)}',
                        flush=True,
                    )

        validation_scores = binary_metrics(
            np.asarray(validation_labels),
            np.asarray(validation_probabilities),
        )
        validation_metrics = {
            'loss': validation_loss_sum / max(validation_seen, 1),
            **validation_scores,
        }
        if not math.isfinite(validation_metrics['pr_auc']):
            raise RuntimeError('Validation PR-AUC is not finite.')

        # ---- History, checkpoint selection, scheduler, and early stopping.
        history['loss'].append(train_metrics['loss'])
        history['auc'].append(train_metrics['auc'])
        history['pr_auc'].append(train_metrics['pr_auc'])
        history['val_loss'].append(validation_metrics['loss'])
        history['val_auc'].append(validation_metrics['auc'])
        history['val_pr_auc'].append(validation_metrics['pr_auc'])
        history['learning_rate'].append(float(optimizer.param_groups[0]['lr']))
        history['optimizer_steps'].append(int(global_step))

        primary_improved = validation_metrics['pr_auc'] > best_pr_auc
        tied_with_better_loss = (
            np.isclose(
                validation_metrics['pr_auc'],
                best_pr_auc,
                rtol=1e-12,
                atol=1e-12,
            )
            and validation_metrics['loss'] < best_validation_loss
        )
        improved = bool(primary_improved or tied_with_better_loss)
        if improved:
            best_pr_auc = float(validation_metrics['pr_auc'])
            best_validation_loss = float(validation_metrics['loss'])
            best_epoch = int(epoch)
            early_stopping_wait = 0
        else:
            early_stopping_wait += 1

        scheduler.step(validation_metrics['pr_auc'])
        save_training_checkpoint(epoch + 1, improved)

        print(
            f"Epoch {epoch}/{epochs} | step {global_step}/{max_optimizer_updates} | "
            f"loss={train_metrics['loss']:.4f} | "
            f"PR-AUC={train_metrics['pr_auc']:.4f} | "
            f"val_loss={validation_metrics['loss']:.4f} | "
            f"val_PR-AUC={validation_metrics['pr_auc']:.4f} | "
            f"best_epoch={best_epoch}"
        )

        if early_stopping_wait >= early_stopping_patience:
            completion_reason = 'early_stopping'
            print(
                f'Early stopping at epoch {epoch}; '
                f'best validation PR-AUC={best_pr_auc:.4f}.'
            )
            break

    if completion_reason is None:
        completion_reason = checkpoint_io.terminal_reason(
            {
                'early_stopping_counter': early_stopping_wait,
                'global_step': global_step,
                'epoch': epochs + 1,
            },
            completion_limits,
        )
    if completion_reason is None:
        raise RuntimeError('Training loop ended without a terminal condition.')

    best_resume_path = checkpoint_io.resume_checkpoint_path(
        CHECKPOINT_DIR, 'checkpoint_best'
    )
    if not best_resume_path.is_file():
        raise RuntimeError('Training completed without a best validated checkpoint.')
    best_payload = checkpoint_io.read_resume_checkpoint(best_resume_path)
    model.load_state_dict(best_payload['model_state_dict'], strict=True)
    best_epoch = best_payload.get('best_epoch', best_epoch)
    del best_payload

    checkpoint_path = CHECKPOINT_DIR / 'checkpoint_best.pt'
    temporary_checkpoint = checkpoint_path.with_name(
        checkpoint_path.name + f'.tmp.{os.getpid()}'
    )
    torch.save(
        {
            'schema_version': 1,
            'architecture': ARCHITECTURE,
            'model_state_dict': checkpoint_io.to_cpu(model.state_dict()),
        },
        temporary_checkpoint,
    )
    os.replace(temporary_checkpoint, checkpoint_path)

    configuration_payload = {
        key: configuration[key]
        for key in ('architecture', 'condition', 'seed', 'gpu')
    }
    configuration_payload.update({
        'experiment_id': configuration['experiment_id'],
        'training_policy': configuration['training_policy_name'],
        'policy_signature': configuration['policy_signature'],
        'dataset_signature': dataset['dataset_metadata']['signature'],
        'completion_reason': completion_reason,
        'training_budget': {
            'max_optimizer_updates': max_optimizer_updates,
            'checkpoint_metric': POLICY['checkpoint_criterion'],
            'scheduler_monitor': POLICY['scheduler_params']['monitor'],
            'early_stopping_monitor': POLICY['early_stopping']['monitor'],
            'effective_batch_size': int(POLICY['effective_batch_size']),
            'validation_interval': validation_interval,
            'validation_manifest': 'data/processed/metadata/val.csv',
        },
        **gpu_execution_metadata,
    })
    atomic_json(RUN_DIR / 'configuration.json', configuration_payload)
    atomic_json(
        RUN_DIR / 'dataset_summary.json',
        {
            **dataset['audit']['full'],
            'validation_manifest': 'data/processed/metadata/val.csv',
            'validation_signature': dataset['dataset_metadata'].get(
                'validation_signature'
            ),
        },
    )
    write_history_csv(RUN_DIR / 'training_history.csv', history)
    source_accounting = accounting_snapshot()
    atomic_json(RUN_DIR / 'source_accounting.json', source_accounting)
    completion_payload = {
        'schema_version': 1,
        'status': 'complete',
        **expected_checkpoint,
        'completion_reason': completion_reason,
        'optimizer_updates_limit': max_optimizer_updates,
        'optimizer_updates_completed': global_step,
        'next_epoch': len(history['loss']) + 1,
        'best_epoch': best_epoch,
        'early_stopping_counter': early_stopping_wait,
        **gpu_execution_metadata,
        'final_checkpoint': 'checkpoint_best.pt',
        'artifacts': list(checkpoint_io.FINAL_ARTIFACTS),
    }
    atomic_json(checkpoint_io.completion_path(RUN_DIR), completion_payload)

    training_result = {
        'status': 'complete',
        'completion_reason': completion_reason,
        'checkpoint': str(checkpoint_path),
        'history': history,
        'resumed_from': resume_source if resume is not None else None,
        'optimizer_updates_limit': max_optimizer_updates,
        'optimizer_updates_completed': global_step,
        'epochs': epochs,
        'best_epoch': best_epoch,
        'source_accounting': source_accounting,
        'output_dir': str(RUN_DIR),
        **gpu_execution_metadata,
    }
    training_result
    return {
        'configuration': configuration,
        'dataset': dataset,
        'training_result': training_result,
    }


job_runs = {condition: {} for condition in CONDITIONS}
for condition in CONDITIONS:
    for seed in SEEDS:
        print(
            f'\n========== {ARCHITECTURE} | {condition} | seed {seed} =========='
        )
        job_runs[condition][seed] = train_one_job(
            condition,
            seed,
            configurations[condition][seed],
            datasets[condition][seed],
        )
        gc.collect()
        if (
            job_runs[condition][seed]['training_result']['status']
            != 'already_complete'
            and torch.cuda.is_available()
        ):
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

training_results = {
    condition: {
        seed: job_runs[condition][seed]['training_result']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
{
    condition: {
        seed: {
            'optimizer_updates_completed': training_results[condition][seed][
                'optimizer_updates_completed'
            ],
            'checkpoint': training_results[condition][seed]['checkpoint'],
            'resumed_from': training_results[condition][seed]['resumed_from'],
        }
        for seed in SEEDS
    }
    for condition in CONDITIONS
}


## 7. Reconstruct validation learning curves from persisted artifacts

The cell reloads each job's recorded history and renders loss and discrimination trajectories using
the same plotting utility. It does not retrain a model or change checkpoint selection. Curves are
diagnostic evidence for optimization stability and early stopping; they are not independent estimates
of generalization performance.

In [ ]:
existing_outputs_by_job = {
    condition: {
        seed: load_existing_outputs(ROOT, configurations[condition][seed])
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
training_history_figures = {
    condition: {
        seed: (
            title_classifier_figure(
                plot_training_history(existing_outputs_by_job[condition][seed]['history']),
                ARCHITECTURE, condition, seed, 'Training history',
            )
            if existing_outputs_by_job[condition][seed]['history']
            else None
        )
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
source_accounting_figures = {
    condition: {
        seed: (
            title_classifier_figure(
                plot_source_accounting(
                    existing_outputs_by_job[condition][seed]['source_accounting']
                ),
                ARCHITECTURE, condition, seed, 'Samples processed by source',
            )
            if existing_outputs_by_job[condition][seed]['source_accounting']
            else None
        )
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
{
    condition: {
        seed: {
            'training_curves': training_history_figures[condition][seed],
            'source_accounting': existing_outputs_by_job[condition][seed][
                'source_accounting'
            ],
            'source_accounting_figure': source_accounting_figures[condition][seed],
        }
        for seed in SEEDS
    }
    for condition in CONDITIONS
}


## 8. Resolve and verify the selected checkpoint for every job

For each condition–seed pair, the code resolves `checkpoint_best.pt` together with its compatibility
metadata. Resolution checks architecture, policy signature, dataset signature, seed, and completion
state so that a checkpoint cannot be silently reused under another experimental condition. The
resulting mapping is the only model source used by subsequent validation inference and attribution
cells.

In [ ]:
existing_outputs_by_job = {
    condition: {
        seed: load_existing_outputs(ROOT, configurations[condition][seed])
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
checkpoints = {
    condition: {
        seed: existing_outputs_by_job[condition][seed]['checkpoint']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
missing_checkpoint_jobs = [
    (condition, seed)
    for condition in CONDITIONS
    for seed in SEEDS
    if checkpoints[condition][seed] is None
]
if missing_checkpoint_jobs:
    raise RuntimeError(
        f'No trained checkpoint is available for jobs {missing_checkpoint_jobs}.'
    )
checkpoints


## 9. Produce or reload validation predictions

Each validation-selected checkpoint is loaded through the architecture adapter and applied to the
unchanged validation rows. Existing prediction artifacts are reused only when their
row alignment remains compatible; otherwise inference is recomputed. Outputs retain image and patient
identifiers, labels, and probabilities, enabling both image-level diagnostics and the prespecified
patient-level analysis.

In [ ]:
existing_outputs_by_job = {
    condition: {
        seed: load_existing_outputs(ROOT, configurations[condition][seed])
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
checkpoints = {
    condition: {
        seed: existing_outputs_by_job[condition][seed]['checkpoint']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
missing_checkpoint_jobs = [
    (condition, seed)
    for condition in CONDITIONS
    for seed in SEEDS
    if checkpoints[condition][seed] is None
]
if missing_checkpoint_jobs:
    raise RuntimeError(
        f'No trained checkpoint is available for jobs {missing_checkpoint_jobs}.'
    )

validation_results = {condition: {} for condition in CONDITIONS}
for condition in CONDITIONS:
    for seed in SEEDS:
        validation_results[condition][seed] = run_validation(
            ROOT,
            configurations[condition][seed],
            datasets[condition][seed],
            checkpoints[condition][seed],
        )
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

validation_rows_by_job = {
    condition: {
        seed: validation_results[condition][seed]['rows']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
validation_metrics_by_job = {
    condition: {
        seed: validation_results[condition][seed]['metrics']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
validation_metrics_by_job


## 10. Summarize validation discrimination and decision performance

The metric cell computes the registered validation summary for all jobs, including ROC-AUC, PR-AUC,
and threshold-dependent measures derived from validation predictions. Threshold estimation and model
selection remain confined to validation. Results are organized by condition and seed so that the
later comparison notebook can construct ensembles without reopening checkpoints or data.

In [ ]:
for condition in CONDITIONS:
    for seed in SEEDS:
        title_classifier_figure(
            plot_validation_curves(
                validation_rows_by_job[condition][seed],
                validation_metrics_by_job[condition][seed]['threshold'],
            ),
            ARCHITECTURE, condition, seed, 'Validation diagnostics',
        )
validation_metrics_by_job


## 11. Assess probability calibration

Reliability diagrams compare predicted probability with observed validation frequency for every
condition–seed job. They are produced from persisted predictions and therefore do not alter training,
thresholds, or checkpoint choice. Calibration plots should be interpreted as finite-sample
diagnostics, especially for the relatively small positive class, rather than as evidence of clinical
calibration in a deployment population.

In [ ]:
calibration_figures = {
    condition: {
        seed: title_classifier_figure(
            plot_calibration(validation_rows_by_job[condition][seed]),
            ARCHITECTURE, condition, seed, 'Validation calibration',
        )
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
calibration_figures


## 12. Inspect errors and fine-tuned–versus–from-scratch disagreements

The analysis constructs reproducible false-positive/false-negative tables and identifies validation
cases with the largest probability disagreement between the two synthetic-data interventions.
Patient and image keys are retained for traceability, while the held-out test split remains
inaccessible. This qualitative analysis is intended to reveal failure modes and hypothesis-generating
patterns; it is not used to revise the registered conditions or decision rules.

In [ ]:
from notebooks.utility.classifier_interpretability import largest_ft_fs_disagreements

error_tables_by_job = {condition: {} for condition in CONDITIONS}
for condition in CONDITIONS:
    for seed in SEEDS:
        job_rows = validation_rows_by_job[condition][seed]
        job_metrics = validation_metrics_by_job[condition][seed]
        error_cases = build_error_case_table(job_rows, job_metrics['threshold'])
        error_tables = {
            'false positives': error_cases[
                error_cases.error_type == 'false_positive'
            ],
            'false negatives': error_cases[
                error_cases.error_type == 'false_negative'
            ],
            'highest-confidence correct predictions': error_cases[
                error_cases.error_type == 'correct'
            ],
        }
        ft_rows = load_prediction_rows(
            ROOT / 'results/3_classifiers/seed_runs' / ARCHITECTURE
            / 'real_plus_best_finetuned_positive' / f'seed_{seed}'
            / 'validation_predictions.csv'
        )
        fs_rows = load_prediction_rows(
            ROOT / 'results/3_classifiers/seed_runs' / ARCHITECTURE
            / 'real_plus_best_fromscratch_positive' / f'seed_{seed}'
            / 'validation_predictions.csv'
        )
        error_tables['largest FT-vs-FS disagreements'] = (
            largest_ft_fs_disagreements(ft_rows, fs_rows)
            if ft_rows and fs_rows
            else None
        )
        error_tables_by_job[condition][seed] = error_tables

error_tables_by_job


## 13. Generate deterministic attribution diagnostics

Validation cases are selected deterministically from the frozen prediction tables, then the corresponding
best checkpoint is reloaded to compute gradient-weighted spatial attribution and integrated
gradients. Raw attribution arrays remain with each experiment, while reusable PNG overlays are saved
under `results/3_classifiers/figures/interpretability/`, independently of the notebook display.
Attribution indicates model sensitivity under
a chosen method; it does not establish causal importance, lesion localization accuracy, or clinical
explanation.

In [ ]:
from notebooks.utility.classifier_interpretability import (
    integrated_gradients,
    select_deterministic_cases,
    torch_spatial_attribution,
)

INTERPRETABILITY_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

interpretability_by_job = {condition: {} for condition in CONDITIONS}
for condition in CONDITIONS:
    for seed in SEEDS:
        interpretability_cases = select_deterministic_cases(
            validation_rows_by_job[condition][seed],
            validation_metrics_by_job[condition][seed]['threshold'],
        )
        if not interpretability_cases:
            raise RuntimeError(
                f'Deterministic validation cases are required for {condition}, seed {seed}.'
            )
        adapter = load_adapter(configurations[condition][seed])
        model = adapter.load_checkpoint(checkpoints[condition][seed]).to(INTERPRETABILITY_DEVICE)

        target_module = model.stages[-1]
        case_loader = adapter.build_validation_dataloader(
            interpretability_cases, seed=seed
        )
        attribution_root = (
            Path(configurations[condition][seed]['checkpoint_dir']) / 'interpretability'
        )
        attribution_root.mkdir(parents=True, exist_ok=True)
        case_index = 0
        for images, _labels in case_loader:
            for image in images:
                category = interpretability_cases[case_index]['category']
                heatmap = torch_spatial_attribution(
                    model, image.unsqueeze(0).to(INTERPRETABILITY_DEVICE), target_module
                )
                np.save(attribution_root / f'{category}_{case_index}.npy', heatmap)
                np.save(attribution_root / f'{category}_{case_index}_ig.npy',
                        integrated_gradients(model, image.unsqueeze(0).to(INTERPRETABILITY_DEVICE)))
                case_index += 1
        interpretability_by_job[condition][seed] = {
            'method': 'gradient-weighted spatial attribution',
            'cases': interpretability_cases,
        }
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

interpretability_by_job


In [ ]:
from notebooks.utility.classifier_interpretability import (
    render_attribution_overlays,
    save_attribution_figure,
)

gradcam_figures = {
    condition: {
        seed: title_classifier_figure(
            render_attribution_overlays(
                interpretability_by_job[condition][seed]['cases'],
                Path(configurations[condition][seed]['checkpoint_dir']) / 'interpretability',
            ),
            ARCHITECTURE, condition, seed, 'Grad-CAM attribution',
        )
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
for condition in CONDITIONS:
    for seed in SEEDS:
        save_attribution_figure(
            gradcam_figures[condition][seed],
            ROOT / 'results/3_classifiers',
            architecture=ARCHITECTURE, condition=condition, seed=seed, method='gradcam',
        )
gradcam_figures

In [ ]:
ig_figures = {
    condition: {
        seed: title_classifier_figure(
            render_attribution_overlays(
                interpretability_by_job[condition][seed]['cases'],
                Path(configurations[condition][seed]['checkpoint_dir']) / 'interpretability',
                suffix='_ig', method='Integrated Gradients',
            ),
            ARCHITECTURE, condition, seed, 'Integrated Gradients attribution',
        )
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
for condition in CONDITIONS:
    for seed in SEEDS:
        save_attribution_figure(
            ig_figures[condition][seed],
            ROOT / 'results/3_classifiers',
            architecture=ARCHITECTURE, condition=condition, seed=seed,
            method='integrated_gradients',
        )
ig_figures